In [ ]:
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
import torchmetrics
from functorch.dim import Tensor
from sklearn.metrics import roc_curve, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from torch import nn, inference_mode
from matplotlib import pyplot as plt
import io
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from IPython.core.display_functions import display
from torchmetrics.functional import recall
import cma


class Model:
    def __init__(self, layers, data, epochs):
        torch.manual_seed(42)
        struct = []
        for i in range(len(layers) - 1):
            torch.manual_seed(42)
            struct.append(('linear' + str(i + 1), nn.Linear(in_features=layers[i], out_features=layers[i + 1])))
            torch.manual_seed(42)
            struct.append(('relu' + str(i + 1), nn.ReLU()))
        torch.manual_seed(42)
        struct.append(('linear' + str(len(layers)), nn.Linear(in_features=layers[-1], out_features=1)))
        torch.manual_seed(42)
        self.model = nn.Sequential(OrderedDict(struct))
        torch.manual_seed(42)
        self.loss_fn = nn.BCEWithLogitsLoss()
        torch.manual_seed(42)
        self.optimazer = torch.optim.Adam(params=self.model.parameters())
        self.epochs = epochs

        self.x_train, self.x_test, self.y_train, self.y_test = data


def normalize_rows(row):
    high = row.max() * 1.00001
    low = row.min() * 0.9999
    normalized_row = (row - low) / (high - low)
    return normalized_row


def read_and_prepare(path: str, label: int):
    df = pd.read_csv(path, sep=';', encoding='utf8')
    df = df.drop(["breakout", "reversal", "single_false_breakout", "consolidation", "trend_follow"], axis=1)
    df.columns = range(df.shape[1])
    df = df.apply(normalize_rows, axis=1)
    df['label'] = label
    return df


def balance_if_needed(df: pd.DataFrame, random_state: int = 42) -> pd.DataFrame:
    counts = df['label'].value_counts()
    if len(counts) < 2:
        return df.reset_index(drop=True)

    major_label = counts.idxmax()
    minor_label = counts.idxmin()
    major_n = counts[major_label]
    minor_n = counts[minor_label]

    major_df = df[df['label'] == major_label].copy()
    minor_df = df[df['label'] == minor_label].copy()

    if major_n > 2 * minor_n:
        repeat_k = max(1, major_n // minor_n)

        minor_expanded = pd.concat([minor_df] * repeat_k, ignore_index=True)

        target_major_n = min(major_n, len(minor_expanded))

        major_trimmed = major_df.sample(n=target_major_n, random_state=random_state)
        # major_trimmed = major_df.head(target_major_n)

        balanced = pd.concat([minor_expanded, major_trimmed], ignore_index=True)
        print(major_n, minor_n)
    else:
        target_n = min(major_n, minor_n)
        major_trimmed = major_df.sample(n=target_n, random_state=random_state)
        minor_trimmed = minor_df.sample(n=target_n, random_state=random_state)
        # major_trimmed = major_df.head(target_n)
        # minor_trimmed = minor_df.head(target_n)
        balanced = pd.concat([major_trimmed, minor_trimmed], ignore_index=True)

    return balanced.sample(frac=1, random_state=random_state).reset_index(drop=True)


def load_dataset(
    train_main_path: str,
    train_secondary_paths: list[str],
    test_main_path: str,
    test_secondary_paths: list[str],
):
    torch.manual_seed(42)

    train_main_df = read_and_prepare(train_main_path, 1)

    train_sec_list = []
    for p in train_secondary_paths:
        train_sec_list.append(read_and_prepare(p, 0))
    train_sec_df = pd.concat(train_sec_list, ignore_index=True)

    test_main_df = read_and_prepare(test_main_path, 1)

    test_sec_list = []
    for p in test_secondary_paths:
        test_sec_list.append(read_and_prepare(p, 0))
    test_sec_df = pd.concat(test_sec_list, ignore_index=True)

    train_df = pd.concat([train_main_df, train_sec_df], ignore_index=True)
    test_df = pd.concat([test_main_df, test_sec_df], ignore_index=True)

    train_df = balance_if_needed(train_df, random_state=55)
    test_df = balance_if_needed(test_df, random_state=55)

    x_train = torch.tensor(train_df.drop('label', axis=1).values, dtype=torch.float32)
    y_train = torch.tensor(train_df['label'].values, dtype=torch.float32)

    x_test = torch.tensor(test_df.drop('label', axis=1).values, dtype=torch.float32)
    y_test = torch.tensor(test_df['label'].values, dtype=torch.float32)

    return x_train, x_test, y_train, y_test


def train_model(model: Model, main_class, sec_class, printing=False):
    torch.manual_seed(42)
    accuracy_calc = torchmetrics.Accuracy(task="binary")
    precision_calc = torchmetrics.Precision(task="binary")
    recall_calc = torchmetrics.Recall(task="binary")
    loss_sum = 0

    min_loss = 1

    train_losses = []
    train_acc = []
    train_prec = []
    train_recall = []
    test_losses = []
    test_acc_list = []
    test_prec_list = []
    test_recall_list = []
    var = []
    train_F1 = []
    test_F1_list = []

    epoch = 0
    epoches = model.epochs
    while epoch < epoches:
        epoch += 1
        torch.manual_seed(42)
        model.model.train()

        y_logits = model.model(model.x_train).squeeze()
        y_pred = torch.round(torch.sigmoid(y_logits))

        # TP, FP, TN, FN, SUP = stat_scores(y_pred, y_train)
        torch.manual_seed(42)
        loss = model.loss_fn(y_logits, model.y_train)
        acc = accuracy_calc(y_pred, model.y_train)
        prec = precision_calc(y_pred, model.y_train)
        recall = recall_calc(y_pred, model.y_train)
        F1 = f1_score(model.y_train.detach().numpy(), y_pred.detach().numpy(), average="binary")
        torch.manual_seed(42)
        model.optimazer.zero_grad()
        torch.manual_seed(42)
        loss.backward()
        torch.manual_seed(42)
        model.optimazer.step()
        torch.manual_seed(42)
        model.model.eval()
        with torch.inference_mode():
            test_logits = model.model(model.x_test).squeeze()
            test_pred = torch.round(torch.sigmoid(test_logits))

            test_loss = model.loss_fn(test_logits, model.y_test)
            test_acc = accuracy_calc(test_pred, model.y_test)
            test_prec = precision_calc(test_pred, model.y_test)
            test_recall = recall_calc(test_pred, model.y_test)
            test_F1 = f1_score(model.y_test.detach().numpy(), test_pred.detach().numpy(), average="binary")

        preds_variance = sum([(0.5 - abs(float(i) - 0.5)) ** 2 for i in torch.sigmoid(y_logits)])
        train_losses += [float(loss)]
        train_acc += [float(acc)]
        train_prec += [float(prec)]
        test_losses += [float(test_loss)]
        test_acc_list += [float(test_acc)]
        test_prec_list += [float(test_prec)]
        var += [preds_variance]
        test_F1_list += [test_F1]
        train_F1 += [F1]
        test_recall_list += [test_recall]
        train_recall += [recall]

        loss_sum += test_loss

        if epoch > 50 and test_loss < min_loss:
            min_loss = test_loss
            best_state = model.model.state_dict()

        if printing and epoch % 10 == 0:
            print(f"e: {epoch} loss: {loss:.5f} prec: {prec * 100:.2f}% t_loss: {test_loss:.5f} t_acc: {test_acc * 100:.2f}%"
                  f" t_prec: {test_prec * 100:.2f}% var: {round(preds_variance, 3)}")

    xx = [i for i in range(epoch)]
    # plt.plot(xx, [min(1, i) for i in train_losses], lw=1, alpha=0.2, color="blue")
    plt.plot(xx, test_F1_list, lw=2, alpha=0.4, color="brown", label="F1")

    plt.plot(xx, [min(1, i) for i in test_losses], lw=2, alpha=0.4, color="blue", label="Losses")

    plt.plot(xx, train_acc, lw=1, alpha=0.2, color="brown")
    plt.plot(xx, test_acc_list, lw=2, alpha=0.3, color="green", label="Accuracy")
    # plt.plot(xx, [min(1, i) for i in var], lw=2, alpha=0.9, color="pink")

    plt.plot(xx, train_prec, lw=2, alpha=0.8, color="yellow")
    plt.plot(xx, test_prec_list, lw=2, alpha=0.5, color="red", label="Precision")
    plt.grid()
    plt.legend()
    plt.savefig("Results/" + str(main_class) + "_" + str(epoch) + "_" + str(round(float(min_loss), 2)) + ".png")
    plt.show()

    plt.figure(figsize=(10, 8))
    fpr, tpr, thresholds = roc_curve(model.y_test, test_logits, pos_label=1)
    plt.plot(fpr, tpr, lw=2, label="ROC")
    plt.plot([0, 1], [0, 1])
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.savefig("Results/" + str(main_class) + "_" + str(sec_class) + "_" + str(epoch) + str(round(float(min_loss), 2)) + "ROC.png")
    plt.show()
    return best_state


def set_model(layers, train_main_path, train_secondary_paths, test_main_path, test_secondary_path, printing=False):
    model = Model(layers, load_dataset(train_main_path, train_secondary_paths, test_main_path, test_secondary_path), 1500, 0)
    state = train_model(model, train_main_path, test_main_path[0], printing)
    model.model.load_state_dict(state)
    return model.model


def training(reversal_T, reversal_V, Consolidation_T, Consolidation_V, trend_T, trend_V, false_T, false_V, breakout_T,
             breakout_V):
    reversal_model = set_model([233, 128, 64], reversal_T, [trend_T, false_T, Consolidation_T, breakout_T],
                                            reversal_V, [trend_V, false_V, breakout_V, Consolidation_V])

    consolidation_model = set_model([233, 128, 64], Consolidation_T, [trend_T, false_T, breakout_T, reversal_T],
                                                 Consolidation_V, [trend_V, false_V, breakout_V, reversal_V])

    trend_model = set_model([233, 128, 64], trend_T, [reversal_T, false_T, breakout_T, Consolidation_T],
                                         trend_V, [reversal_V, false_V, breakout_V, Consolidation_V])

    false_model = set_model([233, 128, 64], false_T, [trend_T, reversal_T, breakout_T, Consolidation_T],
                                         false_V, [trend_V, reversal_V, breakout_V, Consolidation_V])

    breakout_model = set_model([233, 128, 64], breakout_T, [trend_T, false_T, Consolidation_T, reversal_T],
                                            breakout_V, [trend_V, false_V, reversal_V, Consolidation_V])

    r_c = set_model([233, 128, 64], reversal_T, [Consolidation_T], reversal_V, [Consolidation_V])

    r_t = set_model([233, 128, 64], reversal_T, [trend_T], reversal_V, [trend_V])

    r_f = set_model([233, 128, 64], reversal_T, [false_T], reversal_V, [false_V])

    c_t = set_model([233, 128, 64], Consolidation_T, [trend_T], Consolidation_V, [trend_V])

    c_f = set_model([233, 128, 64], Consolidation_T, [false_T], Consolidation_V, [false_V])

    t_f = set_model([233, 128, 64], trend_T, [false_T], trend_V, [false_V])

    r_b = set_model([233, 128, 64], reversal_T, [breakout_T], reversal_V, [breakout_V])

    c_b = set_model([233, 128, 64], Consolidation_T, [breakout_T], Consolidation_V, [breakout_V])

    t_b = set_model([233, 128, 64], trend_T, [breakout_T], trend_V, [breakout_V])

    f_b = set_model([233, 128, 64], false_T, [breakout_T], false_V, [breakout_V])

    ovr_names = ["reversal", "consolidation", "trend", "false", "breakout"]
    ovr_models = [reversal_model, consolidation_model, trend_model, false_model, breakout_model]
    
    ovo_names = ["r_c", "r_t", "r_f", "c_t", "c_f", "t_f", "r_b", "c_b", "t_b", "f_b"]
    ovo_models = [r_c, r_t, r_f, c_t, c_f, t_f, r_b, c_b, t_b, f_b]

    for i in range(len(ovo_names)):
        torch.save(ovo_models[i], ovo_names[i] + "_0" + ".pth")

    for i in range(len(ovr_names)):
        torch.save(ovr_models[i], ovr_names[i] + "_0" + ".pth")


def load_test_dataset(secondary_classes, size=0):
    lens = []
    final_dataset = pd.DataFrame()
    counter = 0
    for p in secondary_classes:
        df = pd.read_csv(p, sep=';', encoding='utf8')
        df = df.drop(["breakout", "reversal", "single_false_breakout", "consolidation", "trend_follow"], axis=1)
        if size != 0:
            if len(df) < 800:
                df = pd.concat([df] * (800 // len(df)), ignore_index=True)
            df = df.head(800)
        df.columns = range(df.shape[1])
        df = df.apply(normalize_rows, axis=1)
        final_dataset = pd.concat([final_dataset, df], ignore_index=True, sort=False)
        counter += 1
        lens += [len(df)]

    final_dataset = final_dataset.apply(normalize_rows, axis=1)
    x = torch.tensor(final_dataset.values).type(torch.float)
    y = []
    for i in range(len(secondary_classes)):
        y += [i] * lens[i]
    return x, y


ovo_models = []
ovo_names = ["r_c", "r_t", "r_f", "c_t", "c_f", "t_f", "r_b", "c_b", "t_b", "f_b"]

ovr_names = ["reversal", "consolidation", "trend", "false", "breakout"]
ovr_models = []

for i in range(len(ovo_names)):
    with open(ovo_names[i] + "_0" + ".pth", "rb") as f:
        buffer = io.BytesIO(f.read())
    ovo_models.append(torch.load(buffer, weights_only=False))

for i in range(len(ovr_names)):
    with open(ovr_names[i] + "_0" + ".pth", "rb") as f:
        buffer = io.BytesIO(f.read())
    ovr_models.append(torch.load(buffer, weights_only=False))

x, y_pre = load_test_dataset(["reversal_0_Val_200.csv", "consolidation_0_Val_3000.csv", "trend_0_Val_1000.csv",
                                  "false_0_Val_400.csv", "breakout_0_Val_200.csv"], 800)

In [ ]:
def classification_OvO(data, models):
    pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (0, 4), (1, 4), (2, 4), (3, 4)]
    N = data.shape[0]

    preds = {pair: np.zeros(N, dtype=int) for pair in pairs}
    for (pair, model) in zip(pairs, models):
        logits = model(data)
        probs = torch.sigmoid(logits).detach().numpy().flatten()
        for i in range(len(probs)):
            preds[pair] = np.round(probs).astype(int)
    votes = np.zeros((N, 5), dtype=int)

    for (i, j) in pairs:
        arr = np.asarray(preds[(i, j)])
        votes[:, i] += (arr == 1)
        votes[:, j] += (arr == 0)
    #
    final = np.zeros(N, dtype=int)

    for k in range(N):
        max_votes = votes[k].max()
        winners = np.where(votes[k] == max_votes)[0]
        if len(winners) == 1:
            final[k] = winners[0]
        elif len(winners) == 2:
            a, b = sorted(winners)
            tie_arr = np.asarray(preds[(a, b)])
            if tie_arr[k] == 1:
                final[k] = a
            else:
                final[k] = b
        else:
            final[k] = int(winners.min())

    return final


preds_pre = classification_OvO(x, ovo_models)
y = torch.from_numpy(np.array(y_pre)).type(torch.int)
preds = torch.from_numpy(np.array(preds_pre)).type(torch.float)

labels = [1, 2, 3, 4, 5]
cm = confusion_matrix(y_pre, preds_pre)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues")
plt.show()

accuracy_calc = torchmetrics.Accuracy(task="multiclass", num_classes=5, average="macro")
precision_calc = torchmetrics.Precision(task="multiclass", num_classes=5, average="macro")
print(accuracy_calc(preds, y))
print(precision_calc(preds, y))

In [ ]:
def classification_Empty(data, models, win_thr=0.6, reject_thr=0.59):
    pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (0, 4), (1, 4), (2, 4), (3, 4)]

    n_classes = 5
    N = data.shape[0]

    votes = np.zeros((N, n_classes), dtype=int)
    alive = np.ones((N, n_classes), dtype=bool)

    with torch.no_grad():
        for (i, j), model in zip(pairs, models):
            logits = model(data)
            p = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1) 

            p_i = p
            p_j = 1.0 - p
            alive[:, i] &= (p_i >= reject_thr)
            alive[:, j] &= (p_j >= reject_thr)
            votes[:, i] += (p_i >= win_thr).astype(int)
            votes[:, j] += (p_j >= win_thr).astype(int)

    final = np.full(N, 5, dtype=int)

    for k in range(N):
        candidates = np.where(alive[k])[0]

        if len(candidates) == 0:
            continue

        cand_votes = votes[k, candidates]
        max_votes = cand_votes.max()

        if max_votes == 0:
            continue

        winners = candidates[cand_votes == max_votes]

        if len(winners) == 1:
            final[k] = int(winners[0])

    return final


preds_pre = classification_Empty(x, ovo_models)
y = torch.from_numpy(np.array(y_pre)).type(torch.int)
preds = torch.from_numpy(np.array(preds_pre)).type(torch.float)

labels = [1, 2, 3, 4, 5, "Empty"]
cm = confusion_matrix(y_pre, preds_pre)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues")
plt.show()

In [ ]:
def metric(true_labels, pred_labels, epsilon=1e-8):
    n_classes = 5
    conf_matrix = np.zeros((n_classes, n_classes + 1), dtype=int)

    for true, pred in zip(true_labels, pred_labels):
        conf_matrix[true, pred] += 1
    scores = []
    for c in range(conf_matrix.shape[0]):
        TP = conf_matrix[c, c]
        FN = conf_matrix[c, :n_classes].sum() - TP
        U = conf_matrix[c, n_classes]
        score_c = np.log(1 + TP / (FN + U + epsilon)) - 6.5 * np.log(1 + FN / (TP + epsilon))
        scores.append(score_c)
    return scores


metric_list = metric(preds_pre, y_pre)
print(metric_list)
print(np.mean(metric_list))

In [ ]:
def classification_full(x, ovo_models, ovr_models, win_thrs, reject_thrs, ovr_reject_weights, ovr_tie_weights):
    if isinstance(x, np.ndarray):
        data = torch.from_numpy(x.astype(np.float32))
    else:
        data = x.float()
    pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (0, 4), (1, 4), (2, 4), (3, 4)]
    n_classes = 5
    N = data.shape[0]
    votes = np.zeros((N, n_classes), dtype=int)
    alive = np.ones((N, n_classes), dtype=bool)
    final = np.full(N, 5, dtype=int)

    # OvO reject
    with torch.no_grad():
        for idx, ((i, j), model) in enumerate(zip(pairs, ovo_models)):
            win_i = win_thrs[2 * idx]
            win_j = win_thrs[2 * idx + 1]
            rej_i = reject_thrs[2 * idx]
            rej_j = reject_thrs[2 * idx + 1]

            logits = model(data)
            p = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
            p_i = p
            p_j = 1.0 - p
            votes[:, i] += (p_i >= win_i).astype(int)
            votes[:, j] += (p_j >= win_j).astype(int)
            alive[:, i] &= (p_i >= rej_i)
            alive[:, j] &= (p_j >= rej_j)

    # OvR reject
    with torch.no_grad():
        ovr_probs = np.zeros((N, n_classes))
        for c, model in enumerate(ovr_models):
            logits = model(data)
            p_c = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
            ovr_probs[:, c] = p_c
            alive[:, c] &= (p_c >= ovr_reject_weights[c])

    # OvR Tie check
    for k in range(N):
        tie_count = np.sum(ovr_probs[k] >= ovr_tie_weights)
        if tie_count >= 2:
            if votes[k, 2] == votes[k].max():
                final[k] = 2
            else:
                final[k] = 5
            continue

        candidates = np.where(alive[k])[0]
        if len(candidates) == 0:
            continue

        cand_votes = votes[k, candidates]
        max_votes = cand_votes.max()

        if max_votes == 0:
            continue

        winners = candidates[cand_votes == max_votes]
        if len(winners) == 1:
            final[k] = int(winners[0])
        else:
            final[k] = 5
    return final


def objective_class(x, data, true_labels, ovo_models, ovr_models, class_idx, old_win=[0.5] * 20, old_rej=[0.0] * 20):
    pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (0, 4), (1, 4), (2, 4), (3, 4)]
    n_classes = 5
    n_pairs = sum([1 for p in pairs if class_idx in p])
    win_thrs_class = x[:n_pairs]
    reject_thrs_class = x[n_pairs:2 * n_pairs]
    ovr_reject_weight_class = x[2 * n_pairs]
    ovr_reject_weight_others = x[2 * n_pairs + 1]

    win_thrs = np.zeros(len(pairs) * 2)
    reject_thrs = np.zeros(len(pairs) * 2)
    counter = 0
    for i, (a, b) in enumerate(pairs):
        if a == class_idx:
            win_thrs[2 * i] = win_thrs_class[counter]
            win_thrs[2 * i + 1] = win_thrs_class[counter] 
            reject_thrs[2 * i] = reject_thrs_class[counter]
            reject_thrs[2 * i + 1] = reject_thrs_class[counter]
            counter += 1
        elif b == class_idx:
            win_thrs[2 * i + 1] = win_thrs_class[counter]
            win_thrs[2 * i] = win_thrs_class[counter]
            reject_thrs[2 * i + 1] = reject_thrs_class[counter]
            reject_thrs[2 * i] = reject_thrs_class[counter]
            counter += 1
        else:
            win_thrs[2 * i] = old_win[2 * i]
            win_thrs[2 * i + 1] = old_win[2 * i + 1]
            reject_thrs[2 * i] = old_rej[2 * i]
            reject_thrs[2 * i + 1] = old_rej[2 * i + 1]

    ovr_reject_weights = np.full(n_classes, ovr_reject_weight_others)
    ovr_reject_weights[class_idx] = ovr_reject_weight_class
    ovr_tie_weights = np.array([0.85, 0.3, 0.65, 0.99, 0.9])

    pred_labels = classification_full(
        data, ovo_models, ovr_models,
        win_thrs, reject_thrs,
        ovr_reject_weights, ovr_tie_weights
    )
    scores = metric(true_labels, pred_labels)
    mean_score = np.mean(scores)
    return -mean_score


def optimize_class(class_idx, data, true_labels, ovo_models, ovr_models,
                   max_iterations=200):
    pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (0, 4), (1, 4), (2, 4), (3, 4)]
    n_pairs = sum([1 for p in pairs if class_idx in p])
    n_vars = 2 * n_pairs + 2
    x0 = np.full(n_vars, 0.5)
    sigma0 = 0.1

    es = cma.CMAEvolutionStrategy(x0, sigma0, {'bounds': [0, 1], 'maxiter': max_iterations})
    best_solution = None
    best_score = -np.inf
    history = []

    while not es.stop():
        solutions = es.ask()
        scores = []
        for s in solutions:
            val = objective_class(s, data, true_labels, ovo_models, ovr_models, class_idx)
            scores.append(val)
            if -val > best_score:
                best_score = -val
                best_solution = s.copy()
        es.tell(solutions, scores)
        # es.disp()
        history.append(best_score)

    return best_solution, best_score, history


def optimize_all_classes(data, true_labels, ovo_models, ovr_models):
    n_classes = 5
    all_solutions = {}
    all_histories = {}
    for c in range(n_classes):
        # print(f"\n=== Optimizing class {c} ===")
        sol, score, history = optimize_class(c, data, true_labels, ovo_models, ovr_models)
        all_solutions[c] = sol
        all_histories[c] = history
    return all_solutions, all_histories


def plot_histories(all_histories):
    plt.figure(figsize=(10, 5))
    for c, h in all_histories.items():
        plt.plot(h, label=f"class {c}")
    plt.xlabel("Iterations")
    plt.ylabel("Best mean metric")
    plt.title("CMA-ES optimization progress")
    plt.legend()
    plt.grid(True)
    plt.show()


data, true_labels = load_test_dataset(
    [reversal_V, Consolidation_V, trend_V, false_V,
        breakout_V])

all_solutions, all_histories = optimize_all_classes(data, true_labels, ovo_models, ovr_models)
plot_histories(all_histories)

pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (0, 4), (1, 4), (2, 4), (3, 4)]
n_classes = 5
win_thrs = np.zeros(len(pairs)*2)
reject_thrs = np.zeros(len(pairs)*2)
ovr_reject_weights = np.zeros(n_classes)
ovr_tie_weights = [0.82, 0.3, 0.69, 1, 0.9]

for class_idx, x in all_solutions.items():
    n_pairs_class = sum([1 for p in pairs if class_idx in p])
    win_class = x[:n_pairs_class]
    reject_class = x[n_pairs_class:2*n_pairs_class]
    ovr_own = x[2*n_pairs_class]
    ovr_others = x[2*n_pairs_class+1]

    ovr_reject_weights[class_idx] = ovr_own
    for i in range(n_classes):
        if i != class_idx:
            ovr_reject_weights[i] = max(ovr_reject_weights[i], ovr_others)

    counter = 0
    for i, (a,b) in enumerate(pairs):
        if a == class_idx:
            win_thrs[2*i] = win_class[counter]
            win_thrs[2*i+1] = win_class[counter]
            reject_thrs[2*i] = reject_class[counter]
            reject_thrs[2*i+1] = reject_class[counter]
            counter += 1
        elif b == class_idx:
            win_thrs[2*i+1] = win_class[counter]
            win_thrs[2*i] = win_class[counter]
            reject_thrs[2*i+1] = reject_class[counter]
            reject_thrs[2*i] = reject_class[counter]
            counter += 1

print([float(_) for _ in win_thrs])
print([float(_) for _ in reject_thrs])
print([float(_) for _ in ovr_reject_weights])
print([float(_) for _ in ovr_tie_weights])

In [ ]:
preds_pre = classification_full(x, ovo_models, ovr_models, win_thrs, reject_thrs, ovr_reject_weights, ovr_tie_weights)
y = torch.from_numpy(np.array(y_pre)).type(torch.int)
preds = torch.from_numpy(np.array(preds_pre)).type(torch.float)

labels = [1, 2, 3, 4, 5, "Empty"]
cm = confusion_matrix(y_pre, preds_pre)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Blues")
plt.show()

metric_list = metric(preds_pre, y_pre)
print(metric_list)
print(np.mean(metric_list))